<a href="https://colab.research.google.com/github/ilmi64/AI-Revenue-Division-/blob/main/Polybot_Development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


# ==========================================
# POLYBOT — DEVELOPMENT ENVIRONMENT
# Phase 1:
# ==========================================

from pathlib import Path
from datetime import datetime
import json
import logging
import platform
import sys

# Project workspace
BASE = Path("/content/PolyBot")

FOLDERS = [
    "data",
    "strategies",
    "research",
    "backtesting",
    "logs",
    "reports",
    "config"
]

for folder in FOLDERS:
    (BASE / folder).mkdir(parents=True, exist_ok=True)

# Safety configuration
CONFIG = {
    "project": "PolyBot",
    "version": "0.1.0",
    "mode": "RESEARCH_AND_PAPER_TRADING",
    "live_trading": False,
    "wallet_signing": False,
    "automatic_orders": False,
    "initial_capital": 0,
    "created": datetime.now().isoformat()
}

config_path = BASE / "config" / "settings.json"

with open(config_path, "w") as file:
    json.dump(CONFIG, file, indent=4)

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(BASE / "logs" / "polybot.log"),
        logging.StreamHandler(sys.stdout)
    ],
    force=True
)

logging.info("PolyBot environment initialized.")

print("\n================================")
print("       POLYBOT INITIALIZED")
print("================================")
print("Python:", platform.python_version())
print("Workspace:", BASE)
print("Folders created:", len(FOLDERS))
print("Operating mode:", CONFIG["mode"])
print("Live trading:", CONFIG["live_trading"])
print("Wallet signing:", CONFIG["wallet_signing"])
print("================================")
print("Foundation setup complete.")

2026-11-14 14:00:22,113 | INFO | PolyBot environment initialized.

       POLYBOT INITIALIZED
Python: 3.13.18
Workspace: /content/PolyBot
Folders created: 7
Operating mode: RESEARCH_AND_PAPER_TRADING
Live trading: False
Wallet signing: False
Foundation setup complete.


In [ ]:

# ==========================================
# POLYBOT — PHASE 2: PERMANENT STORAGE
# ==========================================

from google.colab import drive
from pathlib import Path
import shutil

# Connect Google Drive
drive.mount('/content/drive')

# Permanent project location
DRIVE_BASE = Path(
    "/content/drive/MyDrive/PolyBot"
)

DRIVE_BASE.mkdir(parents=True, exist_ok=True)

# Preserve project folders and files
shutil.copytree(
    "/content/PolyBot",
    DRIVE_BASE,
    dirs_exist_ok=True
)

print("================================")
print("POLYBOT STORAGE CONNECTED")
print("================================")
print("Permanent location:", DRIVE_BASE)
print("Project backup: SUCCESS")
print("================================")

Mounted at /content/drive
POLYBOT STORAGE CONNECTED
Permanent location: /content/drive/MyDrive/PolyBot
Project backup: SUCCESS


In [ ]:

# ==========================================
# POLYBOT - PHASE 3: MARKET INTELLIGENCE
# ==========================================

import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
import time

PAIRS = ["BTC-USD", "ETH-USD", "SOL-USD"]

LOCAL_DATA = Path("/content/PolyBot/data")
DRIVE_DATA = Path("/content/drive/MyDrive/PolyBot/data")

LOCAL_DATA.mkdir(parents=True, exist_ok=True)
DRIVE_DATA.mkdir(parents=True, exist_ok=True)

def collect_market_data(pair):

    url = f"https://api.exchange.coinbase.com/products/{pair}/candles"

    response = requests.get(
        url,
        params={"granularity": 3600},
        headers={"User-Agent": "PolyBot-Research/0.1"},
        timeout=20
    )

    response.raise_for_status()

    records = response.json()

    if not records:
        raise ValueError("No market data returned")

    df = pd.DataFrame(
        records,
        columns=[
            "timestamp", "low", "high",
            "open", "close", "volume"
        ]
    )

    df["timestamp"] = pd.to_datetime(
        df["timestamp"], unit="s", utc=True
    )

    for col in ["low", "high", "open", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna()
    df = df.sort_values("timestamp")
    df = df.drop_duplicates("timestamp")

    df = df[
        (df["low"] > 0) &
        (df["high"] >= df["low"]) &
        (df["volume"] >= 0)
    ].copy()

    df["pair"] = pair
    df["collected_at"] = datetime.now(
        timezone.utc
    ).isoformat()

    return df


market_data = {}

print("POLYBOT MARKET INTELLIGENCE")
print("Public research only")
print("--------------------------------")

for pair in PAIRS:
    try:
        df = collect_market_data(pair)
        market_data[pair] = df

        filename = f"{pair}_hourly.csv"

        df.to_csv(LOCAL_DATA / filename, index=False)
        df.to_csv(DRIVE_DATA / filename, index=False)

        latest_price = df.iloc[-1]["close"]

        print("SUCCESS:", pair)
        print("Records:", len(df))
        print("Latest close:", latest_price, "USD")

    except Exception as error:
        print("UNAVAILABLE:", pair)
        print("Reason:", str(error))

    time.sleep(1)

print("--------------------------------")
print("Pairs collected:", len(market_data))
print("No trades or transactions performed.")

POLYBOT MARKET INTELLIGENCE
Public research only
--------------------------------
SUCCESS: BTC-USD
Records: 350
Latest close: 66471.21 USD
SUCCESS: ETH-USD
Records: 350
Latest close: 1873.26 USD
SUCCESS: SOL-USD
Records: 350
Latest close: 74.78 USD
--------------------------------
Pairs collected: 3
No trades or transactions performed.


In [ ]:
# ==========================================# POLYBOT - PHASE 3B: CROSS-VENUE COMPARISON# ==========================================# Additive research cell: leaves the Coinbase collector above unchanged.# Public unauthenticated Binance Spot candles only. No orders or transactions.import subprocessimport sysfrom pathlib import Pathfrom datetime import datetime, timezoneimport timeimport jsonimport pandas as pdREPO_URL = "https://github.com/ilmi64/AI-Revenue-Division-.git"REPO_BRANCH = "feature/polybot-binance-public-data"REPO_ROOT = Path("/content/PolyBot-integration")if not REPO_ROOT.exists():    subprocess.run(        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)],        check=True,        capture_output=True,        text=True,    )if str(REPO_ROOT) not in sys.path:    sys.path.insert(0, str(REPO_ROOT))from crypto.polybot.binance_public import get_klinesfrom crypto.polybot.market_normalization import (    compare_candle_series,    normalize_binance_klines,    normalize_coinbase_candles,)from crypto.polybot.data_quality import evaluate_candle_qualityPAIR_MAP = {    "BTC-USD": ("BTCUSDT", "BTC", "USD", "USDT"),    "ETH-USD": ("ETHUSDT", "ETH", "USD", "USDT"),    "SOL-USD": ("SOLUSDT", "SOL", "USD", "USDT"),}comparison_reports = {}quality_reports = {}comparison_collected_at = datetime.now(timezone.utc)print("POLYBOT CROSS-VENUE RESEARCH")print("Coinbase USD baseline vs Binance USDT")print("--------------------------------")for coinbase_pair, mapping in PAIR_MAP.items():    binance_symbol, base_asset, coinbase_quote, binance_quote = mapping    coinbase_df = market_data.get(coinbase_pair)    if coinbase_df is None or coinbase_df.empty:        print("SKIPPED:", coinbase_pair, "- Coinbase baseline unavailable")        continue    try:        coinbase_rows = normalize_coinbase_candles(            coinbase_df.to_dict(orient="records"),            pair=coinbase_pair,            interval="1h",            collected_at=comparison_collected_at,        )        raw_binance = get_klines(binance_symbol, interval="1h", limit=200)        binance_rows = normalize_binance_klines(            raw_binance,            symbol=binance_symbol,            base_asset=base_asset,            quote_asset=binance_quote,            interval="1h",            collected_at=comparison_collected_at,        )        report = compare_candle_series(coinbase_rows, binance_rows)        comparison_reports[coinbase_pair] = report        # Evaluate both feeds over the same bounded overlap window. The Binance        # normalizer excludes unfinished candles; exclude any current Coinbase        # candle beyond that completed overlap rather than treating it as closed.        if not coinbase_rows or not binance_rows:            print("QUALITY UNAVAILABLE:", coinbase_pair, "- one source returned no normalized candles")            continue        coinbase_times = [pd.Timestamp(row["event_time"]) for row in coinbase_rows]        binance_times = [pd.Timestamp(row["event_time"]) for row in binance_rows]        window_start = max(min(coinbase_times), min(binance_times))        window_end = min(max(coinbase_times), max(binance_times))        if window_end < window_start:            print("QUALITY UNAVAILABLE:", coinbase_pair, "- no shared hourly history window")            continue        coinbase_window = [            row for row in coinbase_rows            if window_start <= pd.Timestamp(row["event_time"]) <= window_end        ]        binance_window = [            row for row in binance_rows            if window_start <= pd.Timestamp(row["event_time"]) <= window_end        ]        quality_args = {            "expected_start": window_start.to_pydatetime(),            "expected_end": window_end.to_pydatetime(),            "now": comparison_collected_at,            "interval_seconds": 3600,            "max_age_seconds": 5400,            "minimum_coverage": 0.95,            "comparison_rows": report,            "divergence_threshold_percentage_points": 1.0,            "minimum_comparison_samples": 3,        }        coinbase_quality = evaluate_candle_quality(coinbase_window, **quality_args)        binance_quality = evaluate_candle_quality(binance_window, **quality_args)        quality_reports[coinbase_pair] = {            "coinbase": coinbase_quality,            "binance": binance_quality,        }        quality_rows = []        for venue_label, quality in (            ("coinbase_usd", coinbase_quality),            ("binance_usdt", binance_quality),        ):            quality_rows.append({                "pair": coinbase_pair,                "venue": venue_label,                "source": quality["source"],                "base_asset": quality["base_asset"],                "quote_asset": quality["quote_asset"],                "interval": quality["interval"],                "window_start": quality["window"]["start"],                "window_end": quality["window"]["end"],                "expected_candles": quality["expected_candles"],                "valid_candles": quality["valid_candles"],                "missing_candles": quality["missing_candles"],                "duplicate_records": quality["duplicate_records"],                "malformed_records": quality["malformed_records"],                "incomplete_candles": quality["incomplete_candles"],                "out_of_window_records": quality["out_of_window_records"],                "coverage_percent": quality["coverage_percent"],                "feed_age_seconds": quality["freshness"]["age_seconds"],                "feed_stale": quality["freshness"]["stale"],                "usable_for_research": quality["usable_for_research"],                "quality_flags": ";".join(quality["quality_flags"]),                "missing_ranges_json": json.dumps(quality["missing_ranges"]),                "comparison_samples": quality["comparison"]["valid_samples"],                "comparison_insufficient": quality["comparison"]["insufficient_samples"],                "divergence_flags_json": json.dumps(quality["comparison"]["divergence_flags"]),            })        quality_df = pd.DataFrame(quality_rows)        safe_name = coinbase_pair.replace("-", "_")        quality_filename = f"{safe_name}_data_quality_hourly.csv"        quality_df.to_csv(LOCAL_DATA / quality_filename, index=False)        quality_df.to_csv(DRIVE_DATA / quality_filename, index=False)        for quality in (coinbase_quality, binance_quality):            print(                "QUALITY:",                quality["source"],                quality["quote_asset"],                "coverage:",                str(quality["coverage_percent"]) + "%",                "usable:",                quality["usable_for_research"],                "flags:",                quality["quality_flags"] or "none",            )        if report:            report_df = pd.DataFrame(report)            safe_name = coinbase_pair.replace("-", "_")            filename = f"{safe_name}_cross_venue_hourly.csv"            report_df.to_csv(LOCAL_DATA / filename, index=False)            report_df.to_csv(DRIVE_DATA / filename, index=False)            latest = report[-1]            print("SUCCESS:", coinbase_pair, "vs", binance_symbol)            print("Aligned candles:", len(report))            print("Latest Coinbase return (%):", latest["left_return_percent"])            print("Latest Binance return (%):", latest["right_return_percent"])            print("Return difference (percentage points):", latest["return_difference_percentage_points"])            print("Price levels comparable:", latest["price_levels_comparable"])        else:            print("NO OVERLAP:", coinbase_pair, "- no matching completed hourly candles")    except Exception as error:        print("UNAVAILABLE:", coinbase_pair)        print("Reason:", str(error))    time.sleep(1.2)print("--------------------------------")print("Pairs compared:", len(comparison_reports))print("Pairs with quality reports:", len(quality_reports))print("Quality CSVs saved locally and to Google Drive.")print("USD and USDT price levels were not merged or FX-converted.")print("Research only: no orders, wallet access, or transactions.")